# Conquer Online Mobile — Android Manifest Audit
One-code-cell Colab notebook. Downloads and verifies the canonical extracted asset archive, extracts all four identified APK files, decodes Android binary manifests, reports package/version/build/SDK/application/launcher metadata, and uploads JSON, CSV and decoded XML manifests directly to the existing game repository. CPU only; no GPU required. Requires Colab Secret `GITHUB_TOKEN` with repository Contents read/write.

In [ ]:
import os,sys,subprocess,requests,zipfile,hashlib,json,csv,pathlib,io,base64,re
from datetime import datetime,timezone
from google.colab import userdata
ROOT=pathlib.Path("/content/conquer_android_manifest_audit"); ROOT.mkdir(parents=True,exist_ok=True)
TOKEN=userdata.get("GITHUB_TOKEN")
if not TOKEN: raise RuntimeError("Missing Colab Secret GITHUB_TOKEN.")
REPO="ZzwahwahzZ/Clash-of-Legends"; BRANCH="main"; DEST="research/original-client-metadata-audit"
H={"Authorization":f"Bearer {TOKEN}","Accept":"application/vnd.github+json","X-GitHub-Api-Version":"2022-11-28"}
u=requests.get("https://api.github.com/user",headers=H,timeout=40); u.raise_for_status()
r=requests.get(f"https://api.github.com/repos/{REPO}",headers=H,timeout=40); r.raise_for_status()
if not r.json().get("permissions",{}).get("push",False): raise RuntimeError("Token lacks push permission to "+REPO)
print("GITHUB WRITE ACCESS VERIFIED:",REPO,BRANCH)
# Androguard decodes Android binary XML manifests without requiring an Android SDK.
subprocess.run([sys.executable,"-m","pip","-q","install","androguard==4.1.3"],check=True)
from androguard.core.apk import APK
archive=ROOT/"extracted-assets-001.zip"
url="https://github.com/ZzwahwahzZ/Conquer-Mobile-Assets./releases/download/mobile-extracted-assets/extracted-assets-001.zip"
expected_size=842723106; expected_sha="0d21a110c09b762ed5f1a5f4da022857b0534e489eb85052d15da460cb229d2a"
if not archive.exists() or archive.stat().st_size!=expected_size:
    tmp=archive.with_suffix(".part")
    with requests.get(url,stream=True,timeout=(30,240)) as q:
        q.raise_for_status()
        with open(tmp,"wb") as f:
            for chunk in q.iter_content(8*1024*1024):
                if chunk:f.write(chunk)
    tmp.replace(archive)
h=hashlib.sha256()
with open(archive,"rb") as f:
    for chunk in iter(lambda:f.read(8*1024*1024),b""):h.update(chunk)
if archive.stat().st_size!=expected_size or h.hexdigest()!=expected_sha:
    raise RuntimeError(f"Source archive integrity check failed: size={archive.stat().st_size}, sha256={h.hexdigest()}")
print("SOURCE ARCHIVE VERIFIED:",archive.stat().st_size,h.hexdigest())
targets=[
"ConquerOnline/ConquerOnline-1.1.2.3.apk",
"ConquerOnline/config.arm64_v8a.apk",
"ConquerOnline/config.xxhdpi.apk",
"ConquerOnline/install_time_1001.apk"]
extracted=[]
with zipfile.ZipFile(archive) as z:
    names=set(z.namelist())
    for member in targets:
        if member not in names: raise FileNotFoundError("Expected APK missing from source archive: "+member)
        dest=ROOT/pathlib.PurePosixPath(member).name
        with z.open(member) as src,open(dest,"wb") as out:
            while True:
                b=src.read(8*1024*1024)
                if not b:break
                out.write(b)
        extracted.append((member,dest,z.getinfo(member).file_size))
print("EXTRACTED APK FILES:",len(extracted))
def safe_call(obj,name,*args):
    try:
        v=getattr(obj,name)(*args)
        return v if v not in ("",None) else None
    except Exception as e:return {"error":str(e)[:300]}
results=[]; xml_dir=ROOT/"decoded_manifests"; xml_dir.mkdir(exist_ok=True)
for member,p,size in extracted:
    print("\nPARSING:",member,"bytes:",size)
    item={"archive_path":member,"file":p.name,"bytes":size,"sha256":hashlib.sha256(p.read_bytes()).hexdigest()}
    try:
        a=APK(str(p))
        item.update({
          "package_name":safe_call(a,"get_package"),
          "version_name":safe_call(a,"get_androidversion_name"),
          "version_code":safe_call(a,"get_androidversion_code"),
          "version_code_string":safe_call(a,"get_androidversion_code"),
          "min_sdk":safe_call(a,"get_min_sdk_version"),
          "target_sdk":safe_call(a,"get_target_sdk_version"),
          "max_sdk":safe_call(a,"get_max_sdk_version"),
          "app_label":safe_call(a,"get_app_name"),
          "permissions":safe_call(a,"get_permissions"),
          "main_activity":safe_call(a,"get_main_activity"),
          "activities":safe_call(a,"get_activities"),
          "services":safe_call(a,"get_services"),
          "receivers":safe_call(a,"get_receivers"),
          "providers":safe_call(a,"get_providers"),
          "is_valid_APK":safe_call(a,"is_valid_APK"),
          "is_signed":safe_call(a,"is_signed")
        })
        try:
            xml=a.get_android_manifest_xml()
            import xml.etree.ElementTree as ET
            xml_bytes=ET.tostring(xml,encoding="utf-8",xml_declaration=True)
            out=xml_dir/(p.stem+"-AndroidManifest.xml")
            out.write_bytes(xml_bytes); item["decoded_manifest_file"]=out.name
            ns={"a":"http://schemas.android.com/apk/res/android"}
            app=xml.find("application")
            if app is not None:
                item["application_attributes"]=dict(app.attrib)
            acts=[]
            for act in xml.findall(".//activity")+xml.findall(".//activity-alias"):
                rec={"tag":act.tag.split("}")[-1],"name":act.get("{http://schemas.android.com/apk/res/android}name"),"exported":act.get("{http://schemas.android.com/apk/res/android}exported")}
                filters=[]
                for f in act.findall("intent-filter"):
                    filters.append({"actions":[x.get("{http://schemas.android.com/apk/res/android}name") for x in f.findall("action")],"categories":[x.get("{http://schemas.android.com/apk/res/android}name") for x in f.findall("category")]})
                rec["intent_filters"]=filters
                if filters:acts.append(rec)
            item["launchable_activity_candidates"]=acts
        except Exception as e:item["manifest_decode_error"]=str(e)[:600]
    except Exception as e:item["parse_error"]=str(e)[:1000]
    results.append(item)
    print("PACKAGE:",item.get("package_name"),"| VERSION:",item.get("version_name"),"| CODE:",item.get("version_code"),"| MIN SDK:",item.get("min_sdk"),"| TARGET SDK:",item.get("target_sdk"),"| MAIN:",item.get("main_activity"),"| ERROR:",item.get("parse_error") or item.get("manifest_decode_error"))
# Cross-check all APK manifests; do not treat split APK version fields as a separate client release.
base=[x for x in results if x["file"]=="ConquerOnline-1.1.2.3.apk"]
report={"generated_utc":datetime.now(timezone.utc).isoformat(),"source_archive":{"name":archive.name,"bytes":archive.stat().st_size,"sha256":expected_sha,"verified":True},"apk_count":len(results),"apk_manifests":results,"base_apk_assessment":base[0] if base else None,"notes":["The filename 1.1.2.3 is not accepted as the app version unless AndroidManifest confirms it.","Configuration APKs may be split packages; their manifest metadata must be interpreted alongside the base APK.","A manifest version identifies this supplied client package, not necessarily the server protocol/build date."]}
json_path=ROOT/"android-manifest-audit.json"; json_path.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding="utf-8")
csv_path=ROOT/"android-manifest-audit.csv"
fields=["file","archive_path","bytes","sha256","package_name","version_name","version_code","min_sdk","target_sdk","max_sdk","app_label","main_activity","is_valid_APK","is_signed","parse_error","manifest_decode_error"]
with open(csv_path,"w",newline="",encoding="utf-8-sig") as f:
    w=csv.DictWriter(f,fieldnames=fields);w.writeheader()
    for x in results:w.writerow({k:(json.dumps(x.get(k),ensure_ascii=False) if isinstance(x.get(k),(dict,list)) else x.get(k)) for k in fields})
# Package decoded XML manifests with the audit JSON/CSV.
manifest_zip=ROOT/"decoded-android-manifests.zip"
with zipfile.ZipFile(manifest_zip,"w",zipfile.ZIP_DEFLATED) as z:
    for p in xml_dir.glob("*.xml"):z.write(p,p.name)
def publish(p):
    data=p.read_bytes(); path=f"{DEST}/{p.name}"; api=f"https://api.github.com/repos/{REPO}/contents/{path}"
    old=requests.get(api,headers=H,params={"ref":BRANCH},timeout=60)
    if old.status_code==200: sha=old.json()["sha"]
    elif old.status_code==404: sha=None
    else: raise RuntimeError(f"Cannot inspect GitHub destination ({old.status_code}): {old.text[:900]}")
    body={"message":f"Add Android manifest audit: {p.name}","content":base64.b64encode(data).decode(),"branch":BRANCH}
    if sha:body["sha"]=sha
    put=requests.put(api,headers=H,json=body,timeout=120)
    if not put.ok:raise RuntimeError(f"GitHub upload failed for {p.name} ({put.status_code}): {put.text[:1200]}")
    remote=requests.get(api,headers=H,params={"ref":BRANCH},timeout=60);remote.raise_for_status()
    obj=remote.json(); raw=base64.b64decode(obj["content"].replace("\n","")+"===") if obj.get("content") else requests.get(obj["download_url"],timeout=60).content
    local_hash=hashlib.sha256(data).hexdigest();remote_hash=hashlib.sha256(raw).hexdigest()
    if local_hash!=remote_hash:raise RuntimeError(f"SHA256 mismatch for {p.name}: local={local_hash}, remote={remote_hash}")
    print("GITHUB UPLOAD + SHA256 VERIFIED:",p.name,len(data),local_hash,obj["html_url"])
for p in [json_path,csv_path,manifest_zip]:publish(p)
print("\nMANIFEST AUDIT COMPLETE. All reports and decoded manifests are on GitHub.")
